#### **Import libraries**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
import pickle

#### **Loading processed data**

In [2]:
df = pd.read_csv('../data/preprocessed/full_preprocessed.csv')

In [3]:
target = "log_price"

features = [
    "area",
    "front_width",
    "n_bedrooms",
    "n_bathrooms",
    "n_floors",
    "has_interior_info",

    "is_apartment",
    "is_house_facing",
    "is_house_alley",
    "is_villa",

    "interior_score",
    "legal_score"
]

#### **Split the data into training and testing**

In [4]:
X = df[features]
y = df[target]
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size = 0.2, random_state = 42)

print(X.shape)
print(y.shape)
print(train_X.shape)
print(train_y.shape)
print(test_X.shape)
print(test_y.shape)

(9486, 12)
(9486,)
(7588, 12)
(7588,)
(1898, 12)
(1898,)


### **GridsearchCV templates**

In [5]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

def perform_gridsearch(X, y, param_grids, model, cv = 9, scorer = ['r2', 'neg_mean_squared_error']):
    
    kf = KFold(n_splits = cv, shuffle = True, random_state = 42)
    grid_search = GridSearchCV(estimator = model, param_grid = param_grids, cv = kf, scoring = scorer, refit = 'r2', n_jobs=-1)
    grid_search.fit(X, y)

    best_params = grid_search.best_params_
    results = grid_search.cv_results_
    best_models = grid_search.best_estimator_

    best_scores = {}
    for score_name in scorer:
        if score_name == 'r2':
            best_scores['best_r2_score'] = results['mean_test_r2'][results['rank_test_r2'] == 1][0]
        elif score_name == 'neg_mean_squared_error':
            best_scores['best_mse_score'] = -results['mean_test_neg_mean_squared_error'][results['rank_test_neg_mean_squared_error'] == 1][0]

    return {
        'best_params': best_params,
        'scores': best_scores,
        'best_model': best_models
    }

#### **Perform GridSearch for Random Forest**


In [6]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()

param_grids = [
    {
        "bootstrap": [True],
        "max_samples": [0.5, 0.75],
        "n_estimators": [200, 300, 500, 750],
        "max_depth": [10, 20, 30, None],
        "max_features": ["sqrt", "log2"],
        "min_samples_leaf": [1, 2, 3, 4]
    },
    {
        "bootstrap": [False],
        "max_samples": [None],
        "n_estimators": [200, 300, 500, 750],
        "max_depth": [10, 20, 30, None],
        "max_features": ["sqrt", "log2"],
        "min_samples_leaf": [1, 2, 3, 4]
    }
]

result = perform_gridsearch(train_X, train_y, param_grids, model, cv = 5)

with open('rf_model.pkl', 'wb') as f:
    pickle.dump(result, f)
with open('rf_model.pkl', 'rb') as f:
    exp = pickle.load(f)
loaded_model = exp['best_model']
y_test_pred = loaded_model.predict(test_X)
print(f"R2-score on test set: {r2_score(test_y, y_test_pred)}")
print(f"MSE on test set: {mean_squared_error(test_y, y_test_pred)}")
print(f"Best parameters: {exp['best_params']}")
print(f"Best R2-score: {exp['scores']['best_r2_score']}")
print(f"Best MSE score: {exp['scores']['best_mse_score']}")

/opt/homebrew/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


R2-score on test set: 0.7226348179151874
MSE on test set: 0.25404709131125625
Best parameters: {'bootstrap': True, 'max_depth': 20, 'max_features': 'log2', 'max_samples': 0.5, 'min_samples_leaf': 1, 'n_estimators': 300}
Best R2-score: 0.7145023211288806
Best MSE score: 0.27647492428448195
